# 02 · Ridge task expertise (Section 3.1.2, Table D5)

Predicts years of education for individual O\*NET tasks with a Ridge regression on sentence embeddings of the task text, trained on BLS occupation-level education (inherited by every task of the occupation). The model is then applied to the **212 register tasks**, whose 10 occupations are held out of training entirely.

Stages: (1) merge O\*NET tasks with BLS Table 5.4 education; (2) identify the register tasks and their occupations; (3) training pool = all tasks **minus** those 10 occupations; (4) embed with `all-mpnet-base-v2`; (5) GroupKFold CV by occupation; (6) naive random KFold for the leakage gap; (7) α grid search; (8) occupation bootstrap; (9) random held-out-domain check; (10) final fit + occupation-aggregation check; (11) predict the register tasks; (12) validation summary.

| Output (`outputs/02_ridge_expertise/`) | Contents |
|---|---|
| `ridge_task_complexity.csv` | predicted task years, occupation BLS years, bootstrap SD / 95% CI for the 212 tasks → read by 03 (`ridge_task_years`) |
| `tableD5_ridge_diagnostics.csv` | every value in Table D5 (one row per diagnostic) |
| `validation_summary.json` | the same diagnostics as JSON |
| `alpha_sensitivity.csv` | grouped-CV R² for each of the 25 α values |
| `cv_fold_metrics.csv` | per-fold R², RMSE, MAE, Spearman (grouped and random KFold) |
| `training_pool.csv`, `r2r_holdout_tasks.csv` | the two task sets after Stage 3 |
| `embeddings/` | cached embeddings (`.npy`) + the task ids they belong to |

Reads: `inputs/final_task_register.csv`, `inputs/onet/task_statements.xlsx`, `inputs/bls/BLS_education.xlsx`. Independent of 01.

**Runtime.** Embedding the 18,005 training tasks takes 10–20 min on a laptop CPU, so the embeddings are cached in `outputs/02_ridge_expertise/embeddings/` and reused when the task ids match. Set `RECOMPUTE_EMBEDDINGS = True` to force a fresh encode. The 200-sample bootstrap takes a few minutes more.

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

import prep_common as pc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Configuration

In [2]:
# BLS "Typical education needed for entry" -> conventional years of schooling (Section 3.1.2)
BLS_SOC_COL = "2024 National Employment Matrix code"
BLS_EDU_COL = "Typical education needed for entry"
BLS_YEARS_MAP = {
    "No formal educational credential": 10,
    "High school diploma or equivalent": 12,
    "Some college, no degree": 13,
    "Postsecondary nondegree award": 13,
    "Associate's degree": 14,
    "Bachelor's degree": 16,
    "Master's degree": 18,
    "Doctoral or professional degree": 20,
}

EMBEDDING_MODEL = "all-mpnet-base-v2"
RANDOM_SEED = 42
N_SPLITS = 5
N_BOOTSTRAP = 200
ALPHA_GRID = np.logspace(-2, 4, 25)   # 0.01 to 10,000, log-spaced
RECOMPUTE_EMBEDDINGS = False          # True = ignore the cache and re-encode

np.random.seed(RANDOM_SEED)
EMB_DIR = pc.out_path("ridge", "embeddings")
EMB_DIR.mkdir(exist_ok=True)

## Stages 1–3 – Merge O\*NET with BLS education, hold out the 10 register occupations

In [3]:
pc.banner("STAGE 1: Loading and merging O*NET tasks with BLS education")
onet = pc.load_onet_tasks()
bls = pd.read_excel(pc.BLS_EDUCATION_FILE)

# O*NET-SOC has a decimal suffix (11-1011.00); BLS Table 5.4 uses plain 6-digit SOC (11-1011).
onet["SOC_code"] = pc.soc6(onet[pc.ONET_SOC_COL])
bls["SOC_code"] = bls[BLS_SOC_COL].astype(str).str.strip()
bls["expected_years_education"] = bls[BLS_EDU_COL].map(BLS_YEARS_MAP)
unmapped_bls = bls[bls["expected_years_education"].isna()]
if len(unmapped_bls) > 0:
    print(f"  WARNING: {len(unmapped_bls)} BLS rows had an education category not in BLS_YEARS_MAP:",
          unmapped_bls[BLS_EDU_COL].unique())

task_data = onet.merge(bls[["SOC_code", "expected_years_education"]], on="SOC_code", how="left")
n_total = len(task_data)
n_unmatched = task_data["expected_years_education"].isna().sum()
print(f"  Total O*NET task rows: {n_total}")
print(f"  Unmatched to BLS (no education label): {n_unmatched} ({n_unmatched / n_total:.1%}) -- dropped")
task_data = task_data.dropna(subset=["expected_years_education", pc.ONET_TASK_COL]).copy()
task_data = task_data.drop_duplicates(subset=[pc.ONET_SOC_COL, pc.ONET_TASK_COL])
print(f"  Remaining after dropping unmatched/duplicate rows: {len(task_data)}")
print(f"  Unique occupations: {task_data[pc.ONET_SOC_COL].nunique()}")

pc.banner("STAGE 2: Identifying the register tasks and their occupations")
register = pc.load_register()
r2r_task_ids = set(register["task_id"])
r2r_mask = task_data[pc.ONET_TASKID_COL].isin(r2r_task_ids)
matched_r2r_tasks = task_data[r2r_mask].copy()
print(f"  Register tasks: {len(r2r_task_ids)}, matched to O*NET by task_id: {len(matched_r2r_tasks)}")
if len(matched_r2r_tasks) != len(r2r_task_ids):
    print("  WARNING: unmatched ids:", sorted(r2r_task_ids - set(matched_r2r_tasks[pc.ONET_TASKID_COL]))[:10])
r2r_occupations = set(matched_r2r_tasks[pc.ONET_SOC_COL].unique())
print(f"  Parent occupations: {len(r2r_occupations)} -> {sorted(r2r_occupations)}")

pc.banner("STAGE 3: Building the leakage-free training pool (all tasks of the 10 occupations held out)")
train_pool = task_data[~task_data[pc.ONET_SOC_COL].isin(r2r_occupations)].copy()
print(f"  Full task pool: {len(task_data)} tasks across {task_data[pc.ONET_SOC_COL].nunique()} occupations")
print(f"  Held out: {len(task_data) - len(train_pool)} tasks across {len(r2r_occupations)} occupations")
print(f"  Training/validation pool: {len(train_pool)} tasks across {train_pool[pc.ONET_SOC_COL].nunique()} occupations")
pc.save_csv(train_pool, "ridge", "training_pool.csv")
pc.save_csv(matched_r2r_tasks, "ridge", "r2r_holdout_tasks.csv")

STAGE 1: Loading and merging O*NET tasks with BLS education


[nan]
Length: 1, dtype: str
  Total O*NET task rows: 18796
  Unmatched to BLS (no education label): 579 (3.1%) -- dropped
  Remaining after dropping unmatched/duplicate rows: 18217
  Unique occupations: 893
STAGE 2: Identifying the register tasks and their occupations
  Register tasks: 212, matched to O*NET by task_id: 212
  Parent occupations: 10 -> ['11-1011.00', '11-3031.00', '11-3031.01', '13-1051.00', '13-2011.00', '13-2031.00', '13-2082.00', '15-1242.00', '43-3021.00', '43-3031.00']
STAGE 3: Building the leakage-free training pool (all tasks of the 10 occupations held out)
  Full task pool: 18217 tasks across 893 occupations
  Held out: 212 tasks across 10 occupations
  Training/validation pool: 18005 tasks across 883 occupations
[write] outputs/02_ridge_expertise/training_pool.csv  (18,005 rows)
[write] outputs/02_ridge_expertise/r2r_holdout_tasks.csv  (212 rows)


## Stage 4 – Sentence embeddings (cached)

In [4]:
pc.banner("STAGE 4: Sentence embeddings")


def embed(df, name):
    """Encode task text, reusing embeddings/<name>.npy when it was built for the same task ids."""
    ids = df[pc.ONET_TASKID_COL].astype(str).tolist()
    x_path, id_path = EMB_DIR / f"X_{name}.npy", EMB_DIR / f"X_{name}_task_ids.json"
    if not RECOMPUTE_EMBEDDINGS and x_path.exists() and id_path.exists():
        if json.loads(id_path.read_text()) == ids:
            print(f"  {name}: reusing cached {x_path.name}")
            return np.load(x_path)
        print(f"  {name}: cached ids differ from current task set -> re-encoding")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(EMBEDDING_MODEL)
    print(f"  {name}: encoding {len(df)} tasks with {EMBEDDING_MODEL} ...")
    X = model.encode(df[pc.ONET_TASK_COL].tolist(), show_progress_bar=True, batch_size=64)
    np.save(x_path, X)
    id_path.write_text(json.dumps(ids))
    return X


X_train_pool = embed(train_pool, "train_pool")
X_r2r = embed(matched_r2r_tasks, "r2r")
y_train_pool = train_pool["expected_years_education"].values
groups_train_pool = train_pool[pc.ONET_SOC_COL].values
print(f"  Embedding matrix shape (training pool): {X_train_pool.shape}")
print(f"  Embedding matrix shape (register tasks): {X_r2r.shape}")

STAGE 4: Sentence embeddings
  train_pool: reusing cached X_train_pool.npy
  r2r: reusing cached X_r2r.npy
  Embedding matrix shape (training pool): (18005, 768)
  Embedding matrix shape (register tasks): (212, 768)


## Stages 5–7 – Grouped CV, leakage gap, α selection

In [5]:
fold_rows = []


def evaluate_cv(X, y, groups, alpha, cv_splitter, label):
    """CV with a given Ridge alpha and splitter; returns per-fold metrics."""
    r2s, rmses, maes, spearmans = [], [], [], []
    for k, (train_idx, test_idx) in enumerate(cv_splitter.split(X, y, groups)):
        ridge = Ridge(alpha=alpha, random_state=RANDOM_SEED)
        ridge.fit(X[train_idx], y[train_idx])
        preds = ridge.predict(X[test_idx])
        r2s.append(r2_score(y[test_idx], preds))
        rmses.append(np.sqrt(mean_squared_error(y[test_idx], preds)))
        maes.append(mean_absolute_error(y[test_idx], preds))
        spearmans.append(spearmanr(y[test_idx], preds)[0])
        fold_rows.append({"cv": label, "alpha": alpha, "fold": k + 1, "r2": r2s[-1], "rmse": rmses[-1],
                          "mae": maes[-1], "spearman": spearmans[-1]})
    for name, v in [("R2", r2s), ("RMSE", rmses), ("MAE", maes), ("Spearman", spearmans)]:
        print(f"  [{label}] {name}: mean={np.mean(v):.3f}  sd={np.std(v):.3f}")
    return {"r2": r2s, "rmse": rmses, "mae": maes, "spearman": spearmans}


pc.banner("STAGE 5: GroupKFold cross-validation (grouped by occupation)")
gkf = GroupKFold(n_splits=N_SPLITS)
group_cv_results = evaluate_cv(X_train_pool, y_train_pool, groups_train_pool, alpha=1.0,
                               cv_splitter=gkf, label="GroupKFold (by occupation)")

pc.banner("STAGE 6: Naive random KFold (leakage comparison)")
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
random_cv_results = evaluate_cv(X_train_pool, y_train_pool, None, alpha=1.0, cv_splitter=kf, label="Random KFold")
random_r2 = random_cv_results["r2"]
leakage_gap = np.mean(random_r2) - np.mean(group_cv_results["r2"])
print(f"\n  Leakage gap (random R2 - grouped R2): {leakage_gap:.3f}")

pc.banner("STAGE 7: Alpha selection & sensitivity")
alpha_scores = []
for alpha in ALPHA_GRID:
    res = []
    for train_idx, test_idx in gkf.split(X_train_pool, y_train_pool, groups_train_pool):
        ridge = Ridge(alpha=alpha, random_state=RANDOM_SEED)
        ridge.fit(X_train_pool[train_idx], y_train_pool[train_idx])
        res.append(r2_score(y_train_pool[test_idx], ridge.predict(X_train_pool[test_idx])))
    alpha_scores.append(np.mean(res))
best_alpha_idx = int(np.argmax(alpha_scores))
best_alpha = ALPHA_GRID[best_alpha_idx]
print(f"  Best alpha (grouped CV grid search): {best_alpha:.3f} (grouped R2 = {alpha_scores[best_alpha_idx]:.3f})")
if best_alpha_idx in (0, len(ALPHA_GRID) - 1):
    print("  WARNING: best alpha is at the edge of the search grid -- consider expanding ALPHA_GRID.")
pc.save_csv(pd.DataFrame({"alpha": ALPHA_GRID, "grouped_r2": alpha_scores}), "ridge", "alpha_sensitivity.csv")

print("\n  Re-evaluating GroupKFold metrics at the selected alpha:")
group_cv_results_best_alpha = evaluate_cv(X_train_pool, y_train_pool, groups_train_pool, alpha=best_alpha,
                                          cv_splitter=gkf, label=f"GroupKFold (alpha={best_alpha:.2f})")
pc.save_csv(pd.DataFrame(fold_rows), "ridge", "cv_fold_metrics.csv")

STAGE 5: GroupKFold cross-validation (grouped by occupation)


  [GroupKFold (by occupation)] R2: mean=0.570  sd=0.034
  [GroupKFold (by occupation)] RMSE: mean=1.852  sd=0.065
  [GroupKFold (by occupation)] MAE: mean=1.445  sd=0.045
  [GroupKFold (by occupation)] Spearman: mean=0.751  sd=0.024
STAGE 6: Naive random KFold (leakage comparison)


  [Random KFold] R2: mean=0.608  sd=0.004
  [Random KFold] RMSE: mean=1.772  sd=0.008
  [Random KFold] MAE: mean=1.378  sd=0.005
  [Random KFold] Spearman: mean=0.775  sd=0.004

  Leakage gap (random R2 - grouped R2): 0.038
STAGE 7: Alpha selection & sensitivity


  Best alpha (grouped CV grid search): 1.000 (grouped R2 = 0.570)
[write] outputs/02_ridge_expertise/alpha_sensitivity.csv  (25 rows)

  Re-evaluating GroupKFold metrics at the selected alpha:


  [GroupKFold (alpha=1.00)] R2: mean=0.570  sd=0.034
  [GroupKFold (alpha=1.00)] RMSE: mean=1.852  sd=0.065
  [GroupKFold (alpha=1.00)] MAE: mean=1.445  sd=0.045
  [GroupKFold (alpha=1.00)] Spearman: mean=0.751  sd=0.024
[write] outputs/02_ridge_expertise/cv_fold_metrics.csv  (15 rows)


## Stages 8–9 – Bootstrap stability and held-out-domain check

In [6]:
pc.banner("STAGE 8: Bootstrap stability check")
print(f"  Resampling occupations with replacement, refitting Ridge {N_BOOTSTRAP} times ...")
unique_occs = np.unique(groups_train_pool)
bootstrap_preds = np.zeros((N_BOOTSTRAP, len(X_r2r)))
rng = np.random.RandomState(RANDOM_SEED)
for b in range(N_BOOTSTRAP):
    sampled_occs = rng.choice(unique_occs, size=len(unique_occs), replace=True)
    mask = np.isin(groups_train_pool, sampled_occs)
    ridge_b = Ridge(alpha=best_alpha, random_state=RANDOM_SEED)
    ridge_b.fit(X_train_pool[mask], y_train_pool[mask])
    bootstrap_preds[b] = ridge_b.predict(X_r2r)
boot_std = bootstrap_preds.std(axis=0)
boot_ci_low = np.percentile(bootstrap_preds, 2.5, axis=0)
boot_ci_high = np.percentile(bootstrap_preds, 97.5, axis=0)
print(f"  Mean prediction SD across bootstrap runs (register tasks): {boot_std.mean():.3f} years")
print(f"  Max prediction SD (least stable task): {boot_std.max():.3f} years")

pc.banner("STAGE 9: Held-out-domain sanity check (10 random training occupations)")
rng2 = np.random.RandomState(RANDOM_SEED + 1)
random_holdout_occs = rng2.choice(unique_occs, size=10, replace=False)
holdout_mask = np.isin(groups_train_pool, random_holdout_occs)
ridge_domain_test = Ridge(alpha=best_alpha, random_state=RANDOM_SEED)
ridge_domain_test.fit(X_train_pool[~holdout_mask], y_train_pool[~holdout_mask])
domain_preds = ridge_domain_test.predict(X_train_pool[holdout_mask])
domain_r2 = r2_score(y_train_pool[holdout_mask], domain_preds)
domain_spear = spearmanr(y_train_pool[holdout_mask], domain_preds)[0]
print(f"  Held-out-domain R2: {domain_r2:.3f}   Spearman: {domain_spear:.3f}")

STAGE 8: Bootstrap stability check
  Resampling occupations with replacement, refitting Ridge 200 times ...


  Mean prediction SD across bootstrap runs (register tasks): 0.275 years
  Max prediction SD (least stable task): 0.418 years
STAGE 9: Held-out-domain sanity check (10 random training occupations)


  Held-out-domain R2: 0.530   Spearman: 0.689


## Stages 10–11 – Final model, occupation-aggregation check, predictions for the register tasks

The occupation-aggregation R² is **in-sample** (the final model predicts its own training tasks, averaged by occupation): a sanity check that the task-level model is still anchored to the BLS labels, not an out-of-sample validation.

In [7]:
pc.banner("STAGE 10: Final model on the full training pool")
final_model = Ridge(alpha=best_alpha, random_state=RANDOM_SEED)
final_model.fit(X_train_pool, y_train_pool)
agg_check = pd.DataFrame({"occupation": groups_train_pool, "actual": y_train_pool,
                          "predicted": final_model.predict(X_train_pool)}).groupby("occupation").mean()
agg_r2 = r2_score(agg_check["actual"], agg_check["predicted"])
print(f"  Final Ridge fit on {len(X_train_pool)} tasks, alpha={best_alpha:.3f}")
print(f"  Occupation-level aggregation R2 (training pool, in-sample): {agg_r2:.3f}")

pc.banner("STAGE 11: Predicting the register tasks")
results = matched_r2r_tasks[[pc.ONET_SOC_COL, pc.ONET_TITLE_COL, pc.ONET_TASKID_COL, pc.ONET_TASK_COL,
                             "expected_years_education"]].copy()
results = results.rename(columns={"expected_years_education": "occupation_bls_years"})
results["predicted_task_years"] = final_model.predict(X_r2r)
results["diff_from_occupation_avg"] = results["predicted_task_years"] - results["occupation_bls_years"]
results["bootstrap_sd"] = boot_std
results["bootstrap_ci_low"] = boot_ci_low
results["bootstrap_ci_high"] = boot_ci_high
results = results.sort_values("diff_from_occupation_avg", ascending=False)
pc.save_csv(results, "ridge", "ridge_task_complexity.csv")

show = ["Title", pc.ONET_TASK_COL, "occupation_bls_years", "predicted_task_years"]
print("\n  Tasks scored most ABOVE their occupation's level:")
print(results[show].head(5).to_string(index=False))
print("\n  Tasks scored most BELOW their occupation's level:")
print(results[show].tail(5).to_string(index=False))

STAGE 10: Final model on the full training pool


  Final Ridge fit on 18005 tasks, alpha=1.000
  Occupation-level aggregation R2 (training pool, in-sample): 0.753
STAGE 11: Predicting the register tasks
[write] outputs/02_ridge_expertise/ridge_task_complexity.csv  (212 rows)

  Tasks scored most ABOVE their occupation's level:
        Title                                                                                                         Task  occupation_bls_years  predicted_task_years
Tax Preparers                                             Explain federal and state tax laws to individuals and companies.                  12.0             16.199528
Tax Preparers                 Calculate form preparation fees according to return complexity and processing time required.                  12.0             15.045111
Tax Preparers Interview clients to obtain additional information on taxable income and deductible expenses and allowances.                  12.0             14.909000
Tax Preparers                                       

## Stage 12 – Validation summary and Table D5

In [8]:
summary = {
    "onet_task_rows": int(n_total),
    "onet_rows_unmatched_to_bls": int(n_unmatched),
    "corpus_tasks": int(len(task_data)),
    "corpus_occupations": int(task_data[pc.ONET_SOC_COL].nunique()),
    "training_pool_size": int(len(train_pool)),
    "training_pool_occupations": int(train_pool[pc.ONET_SOC_COL].nunique()),
    "r2r_tasks_held_out": int(len(matched_r2r_tasks)),
    "r2r_occupations_held_out": int(len(r2r_occupations)),
    "cv_folds": N_SPLITS,
    "alpha_grid": f"{len(ALPHA_GRID)} log-spaced values, {ALPHA_GRID[0]:g} to {ALPHA_GRID[-1]:g}",
    "selected_alpha": float(best_alpha),
    "groupkfold_r2_mean": float(np.mean(group_cv_results_best_alpha["r2"])),
    "groupkfold_r2_sd": float(np.std(group_cv_results_best_alpha["r2"])),
    "groupkfold_rmse_mean": float(np.mean(group_cv_results_best_alpha["rmse"])),
    "groupkfold_mae_mean": float(np.mean(group_cv_results_best_alpha["mae"])),
    "groupkfold_spearman_mean": float(np.mean(group_cv_results_best_alpha["spearman"])),
    "random_kfold_r2_mean": float(np.mean(random_r2)),
    "leakage_gap_r2": float(leakage_gap),
    "occupation_aggregation_sanity_r2": float(agg_r2),
    "random_domain_holdout_r2": float(domain_r2),
    "random_domain_holdout_spearman": float(domain_spear),
    "bootstrap_mean_prediction_sd": float(boot_std.mean()),
    "bootstrap_max_prediction_sd": float(boot_std.max()),
}
pc.out_path("ridge", "validation_summary.json").write_text(json.dumps(summary, indent=2))
print("[write] outputs/02_ridge_expertise/validation_summary.json")

D3_LABELS = [
    ("corpus_tasks", "O*NET tasks after merge with BLS"), ("corpus_occupations", "Occupations after merge"),
    ("training_pool_size", "Training pool tasks"), ("training_pool_occupations", "Training pool occupations"),
    ("r2r_tasks_held_out", "Prediction set (register tasks)"), ("r2r_occupations_held_out", "Held-out occupations"),
    ("cv_folds", "GroupKFold folds"), ("alpha_grid", "Alpha grid"), ("selected_alpha", "Selected alpha"),
    ("groupkfold_r2_mean", "Grouped CV R2 (mean)"), ("groupkfold_r2_sd", "Grouped CV R2 (SD)"),
    ("groupkfold_rmse_mean", "Grouped CV RMSE (years)"), ("groupkfold_mae_mean", "Grouped CV MAE (years)"),
    ("groupkfold_spearman_mean", "Grouped CV Spearman rho"), ("random_kfold_r2_mean", "Random KFold R2"),
    ("leakage_gap_r2", "Leakage gap (random - grouped R2)"),
    ("occupation_aggregation_sanity_r2", "Occupation-aggregation R2 (in-sample)"),
    ("random_domain_holdout_r2", "Held-out domain R2 (10 random occupations)"),
    ("random_domain_holdout_spearman", "Held-out domain Spearman"),
    ("bootstrap_mean_prediction_sd", "Bootstrap prediction SD, mean (years)"),
    ("bootstrap_max_prediction_sd", "Bootstrap prediction SD, max (years)"),
]
tableD5 = pd.DataFrame([{"diagnostic": label, "key": k, "value": summary[k]} for k, label in D3_LABELS])
pc.save_csv(tableD5, "ridge", "tableD5_ridge_diagnostics.csv")
tableD5

[write] outputs/02_ridge_expertise/validation_summary.json
[write] outputs/02_ridge_expertise/tableD5_ridge_diagnostics.csv  (21 rows)


,diagnostic,key,value
0,O*NET tasks after merge with BLS,corpus_tasks,18217
1,Occupations after merge,corpus_occupations,893
2,Training pool tasks,training_pool_size,18005
3,Training pool occupations,training_pool_occupations,883
4,Prediction set (register tasks),r2r_tasks_held_out,212
5,Held-out occupations,r2r_occupations_held_out,10
6,GroupKFold folds,cv_folds,5
7,Alpha grid,alpha_grid,"25 log-spaced values, 0.01 to 10000"
8,Selected alpha,selected_alpha,1.0
9,Grouped CV R2 (mean),groupkfold_r2_mean,0.56983
